
# Notebook 2 — Data Wrangling & Cleaning

This notebook takes the **dirty synthetic dataset** created in Notebook 1 and produces a **clean, analysis-ready** file.

**Input**
- `synthetic_sound_the_alarm_2015_2025_dirty.csv`

**Output**
- `synthetic_clean.csv` (short name for convenience)
- `synthetic_sound_the_alarm_2015_2025_clean.csv` (long, descriptive name)

**Cleaning goals (gentle, industry-style):**
1. Fix **nulls** in `fire_escape_plans_num` (impute to 0 for count data).
2. Standardize **categorical typos** in `territory` (e.g., `LAMetro`, `OrangeCounty-X` → canonical names).
3. Handle **outliers** in `smoke_alarms_installed_num` (winsorize upper tail).
4. Review **zero-service rows** (drop most but keep a 10% sample to reflect real logs).
5. Enforce **types & ranges** (dates, integers for counts).
6. Quick **QA checks** before saving.


In [4]:

import numpy as np
import pandas as pd

# Reproducibility for any sampling steps
RANDOM_STATE = 42

# I/O paths
DIRTY_PATH = r"C:\Users\syksh\Documents\community_safety_forecasting\data\synthetic_sound_the_alarm_2015_2025_dirty.csv"
CLEAN_SHORT = r"C:\Users\syksh\Documents\community_safety_forecasting\data\synthetic_clean.csv"
CLEAN_LONG = r"C:\Users\syksh\Documents\community_safety_forecasting\data\synthetic_sound_the_alarm_2015_2025_clean.csv"


## 1) Load dirty dataset

In [5]:

df = pd.read_csv(DIRTY_PATH, dtype={"zip": str})
print("Loaded shape:", df.shape)
df.head()


Loaded shape: (25931, 11)


,date,zip,territory,people_served_num,smoke_alarms_installed_num,fire_escape_plans_num,additional_hazards_num,year,month,dow,is_weekend
0,2017-04-08,93450,Central Coast,2,1,2.0,0,2017,4,5,1
1,2015-08-29,92066,San Diego,7,7,1.0,0,2015,8,5,1
2,2016-03-02,92780,Orange County,8,3,1.0,0,2016,3,2,0
3,2023-05-17,90127,LA Metro,8,6,1.0,1,2023,5,2,0
4,2017-04-23,92316,Inland Empire,8,4,0.0,0,2017,4,6,1


## 2) Quick profile of issues

In [6]:

issues = {}

# Nulls snapshot
null_rates = df.isna().mean().sort_values(ascending=False)
issues['null_rates'] = null_rates

# Territory sample values (to detect typos)
territory_sample = df['territory'].value_counts().head(10)

# Outlier quick look
count_cols = ['people_served_num','smoke_alarms_installed_num','fire_escape_plans_num','additional_hazards_num']
describe_counts = df[count_cols].describe()

issues['territory_sample'] = territory_sample
issues['describe_counts'] = describe_counts

print("Null rates:\n", null_rates, "\n")
print("Territory sample:\n", territory_sample, "\n")
print("Count columns describe:\n", describe_counts)


Null rates:
 fire_escape_plans_num         0.029964
date                          0.000000
zip                           0.000000
territory                     0.000000
people_served_num             0.000000
smoke_alarms_installed_num    0.000000
additional_hazards_num        0.000000
year                          0.000000
month                         0.000000
dow                           0.000000
is_weekend                    0.000000
dtype: float64 

Territory sample:
 territory
LA Metro         7911
Inland Empire    5498
Orange County    4898
San Diego        4295
Central Coast    2811
LAMetro           104
OrangeCounty       70
LA Metro-X         61
SanDiego           50
InlandEmpire       48
Name: count, dtype: int64 

Count columns describe:
        people_served_num  smoke_alarms_installed_num  fire_escape_plans_num  \
count       25931.000000                25931.000000           25154.000000   
mean            4.293741                    3.661872               0.645265   
st

## 3) Cleaning steps

### 3.1 Impute nulls in `fire_escape_plans_num` → 0

In [7]:

if 'fire_escape_plans_num' in df.columns:
    before_nulls = df['fire_escape_plans_num'].isna().sum()
    df['fire_escape_plans_num'] = df['fire_escape_plans_num'].fillna(0)
    after_nulls = df['fire_escape_plans_num'].isna().sum()
    print(f"Imputed fire_escape_plans_num nulls: {before_nulls} -> {after_nulls}")


Imputed fire_escape_plans_num nulls: 777 -> 0


### 3.2 Standardize `territory` strings (fix gentle typos)

In [14]:

# Canonical territory names expected from Notebook 1
CANONICAL_TERRITORIES = [
    "LA Metro",
    "Orange County",
    "Inland Empire",
    "San Diego",
    "Central Coast",
]

def normalize_key(s: str) -> str:
    # Remove non-letters, collapse spaces, lowercase for matching
    keep = ''.join(ch for ch in s if ch.isalpha() or ch.isspace())
    return keep.replace(" ", "").lower()

# Build a lookup keyed by collapsed form (no spaces, lowercase)
canon_lookup = {normalize_key(t): t for t in CANONICAL_TERRITORIES}

def fix_territory(val: str) -> str:
    if not isinstance(val, str):
        return val
    # Remove a trailing '-X' if present, and also remove any hyphens
    v = val.replace("-X", "").replace("-", " ")
    key = normalize_key(v)
    # Try direct canonical match
    if key in canon_lookup:
        return canon_lookup[key]
    # Heuristic fallbacks for common variants
    if key.startswith("lametro"):
        return "LA Metro"
    if key.startswith("orangecounty"):
        return "Orange County"
    if key.startswith("inlandempire"):
        return "Inland Empire"
    if key.startswith("sandiego"):
        return "San Diego"
    if key.startswith("centralcoast"):
        return "Central Coast"
    # If no match, return cleaned original
    return v.strip()

before_unique = sorted(df['territory'].unique().tolist())
df['territory'] = df['territory'].astype(str).apply(fix_territory)
after_unique = sorted(df['territory'].unique().tolist())

print("Territory unique values BEFORE:\n", before_unique[:10])
print("\nTerritory unique values AFTER:\n", after_unique[:10])


Territory unique values BEFORE:
 ['Central Coast', 'Inland Empire', 'LA Metro', 'Orange County', 'San Diego']

Territory unique values AFTER:
 ['Central Coast', 'Inland Empire', 'LA Metro', 'Orange County', 'San Diego']


### 3.3 Winsorize outliers in `smoke_alarms_installed_num`

In [15]:

col = 'smoke_alarms_installed_num'
if col in df.columns:
    # Compute an upper cap using the 99.5th percentile, but not below 40 (since Notebook 1 injected big values >=40)
    upper_pct = df[col].quantile(0.995)
    upper_cap = max(40, int(np.ceil(upper_pct)))
    before_max = df[col].max()
    df[col] = df[col].clip(upper=upper_cap)
    after_max = df[col].max()
    print(f"{col}: winsorized upper tail at {upper_cap}. max {before_max} -> {after_max}")


smoke_alarms_installed_num: winsorized upper tail at 40. max 40 -> 40


### 3.4 Drop most all-zero service rows (keep 10% sample)

In [10]:

count_cols = ['people_served_num','smoke_alarms_installed_num','fire_escape_plans_num','additional_hazards_num']
zero_mask = (df[count_cols].sum(axis=1) == 0)

n_zeros = int(zero_mask.sum())
print("All-zero rows detected:", n_zeros)

# Keep a 10% random sample of zeros, drop the rest
keep_frac = 0.10
np.random.seed(RANDOM_STATE)
keep_indices = df[zero_mask].sample(frac=keep_frac, random_state=RANDOM_STATE).index if n_zeros > 0 else []
drop_mask = zero_mask & ~df.index.isin(keep_indices)

df = df.loc[~drop_mask].reset_index(drop=True)
print("Dropped zeros:", n_zeros - len(keep_indices), "| Kept zeros:", len(keep_indices))


All-zero rows detected: 185
Dropped zeros: 167 | Kept zeros: 18


### 3.5 Enforce types & ranges

In [16]:

# Date -> datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Count columns -> non-negative integers
for c in ['people_served_num','smoke_alarms_installed_num','fire_escape_plans_num','additional_hazards_num']:
    # Clip any negatives (shouldn't exist) and convert to int
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    df[c] = df[c].clip(lower=0).round().astype(int)

# Helper columns (recompute from date for safety)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['dow'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['dow'] >= 5).astype(int)

# Zip as string (no changes to values to avoid introducing inconsistency)
df['zip'] = df['zip'].astype(str)

print(df.dtypes)


date                          datetime64[ns]
zip                                   object
territory                             object
people_served_num                      int32
smoke_alarms_installed_num             int32
fire_escape_plans_num                  int32
additional_hazards_num                 int32
year                                   int32
month                                  int32
dow                                    int32
is_weekend                             int32
dtype: object


## 4) QA checks

In [12]:

print("Final shape:", df.shape)
print("\nNull rates after cleaning:\n", df.isna().mean().sort_values(ascending=False).head())

# Sanity summaries
summary_counts = df[['people_served_num','smoke_alarms_installed_num','fire_escape_plans_num','additional_hazards_num']].describe()
summary_by_year = df.groupby('year')[['people_served_num','smoke_alarms_installed_num']].sum().head()

display(summary_counts)
display(summary_by_year.head())
df.head()


Final shape: (25764, 11)

Null rates after cleaning:
 date                          0.0
zip                           0.0
territory                     0.0
people_served_num             0.0
smoke_alarms_installed_num    0.0
dtype: float64


,people_served_num,smoke_alarms_installed_num,fire_escape_plans_num,additional_hazards_num
count,25764.000000,25764.000000,25764.000000,25764.000000
mean,4.321573,3.685608,0.629988,0.442284
std,4.460711,3.973134,0.792673,0.662489
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000,0.000000
50%,3.000000,3.000000,0.000000,0.000000
75%,6.000000,5.000000,1.000000,1.000000
max,42.000000,40.000000,6.000000,4.000000


,people_served_num,smoke_alarms_installed_num
year,,
2015,9691,8383
2016,10128,8605
2017,10614,8891
2018,10819,9256
2019,11074,9565


,date,zip,territory,people_served_num,smoke_alarms_installed_num,fire_escape_plans_num,additional_hazards_num,year,month,dow,is_weekend
0,2017-04-08,93450,Central Coast,2,1,2,0,2017,4,5,1
1,2015-08-29,92066,San Diego,7,7,1,0,2015,8,5,1
2,2016-03-02,92780,Orange County,8,3,1,0,2016,3,2,0
3,2023-05-17,90127,LA Metro,8,6,1,1,2023,5,2,0
4,2017-04-23,92316,Inland Empire,8,4,0,0,2017,4,6,1


## 5) Save cleaned datasets

In [13]:

df.to_csv(CLEAN_SHORT, index=False)
df.to_csv(CLEAN_LONG, index=False)
print("Saved:", CLEAN_SHORT)
print("Saved:", CLEAN_LONG)


Saved: C:\Users\syksh\Documents\community_safety_forecasting\data\synthetic_clean.csv
Saved: C:\Users\syksh\Documents\community_safety_forecasting\data\synthetic_sound_the_alarm_2015_2025_clean.csv
